# Japan Road Traffic Volume: a first look

Hourly vehicle counts from about 2,000 traffic counters on Japan's national highways
(MLIT Traffic Volume API, reference values, data provided by JARTIC). This notebook loads the
files, shows when traffic peaks, asks a simple question: **was a road busier than usual?**,
and puts the counts next to travel speeds from the 2021 Road Traffic Census.

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# On Kaggle the files sit somewhere under /kaggle/input; elsewhere they are in data/.
found = sorted(glob.glob("/kaggle/input/**/counters.csv", recursive=True)) + ["data/counters.csv"]
DATA = os.path.dirname(next(p for p in found if os.path.exists(p)))
plt.rcParams.update({"axes.titlesize": 16, "axes.labelsize": 14, "legend.fontsize": 12,
                     "xtick.labelsize": 12, "ytick.labelsize": 12})

counters = pd.read_csv(f"{DATA}/counters.csv", dtype={"municipality_code": str})
hourly = pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(f"{DATA}/hourly_*.parquet"))],
                   ignore_index=True)
print(f"{len(hourly):,} hourly rows, {hourly.time_jst.min()} to {hourly.time_jst.max()}")
print(hourly.groupby("sensor").counter_id.nunique().rename("counters"))

## One number per row: vehicles in both directions
Every row has a ready-made `vehicles` column (both directions). Permanent counters (`loop`) add up
small, large and unclassified vehicles; CCTV counters give the total directly. CCTV counts are often
empty (camera off its preset position and other reasons), so this notebook uses permanent counters
only. A permanent counter that fails reports **0 vehicles with a fault or missing flag**; `vehicles`
is left empty for those hours, and `flagged` marks them.

In [ ]:
loop = hourly[hourly.sensor == "loop"].copy()
print(f"rows flagged by the source: {loop.flagged.mean():.1%} (vehicles left empty)")
loop = loop[loop.vehicles.notna()]
for c in ("vehicles", "up_large", "down_large"):
    loop[c] = loop[c].astype("float64")  # nullable integers -> float
loop["share_large"] = (loop.up_large + loop.down_large) / loop.vehicles
print(f"rows with a full count: {len(loop):,}")

## When is traffic heaviest?

In [ ]:
loop["hour"] = loop.time_jst.dt.hour
loop["weekday"] = loop.time_jst.dt.dayofweek
grid = loop.groupby(["weekday", "hour"]).vehicles.mean().unstack()

fig, ax = plt.subplots(figsize=(12, 4.5))
im = ax.imshow(grid, aspect="auto", cmap="magma")
ax.set_yticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
ax.set_xticks(range(0, 24, 3))
ax.set_xlabel("Hour (JST)")
ax.set_title("Mean vehicles per hour at a permanent counter")
fig.colorbar(im, ax=ax, label="vehicles / hour")
plt.tight_layout()
plt.show()

## Trucks at night
The share of large vehicles rises at night, when freight dominates.

In [ ]:
share = loop.groupby("hour").apply(lambda g: g[["up_large", "down_large"]].sum().sum() / g.vehicles.sum())
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(share.index, share.values * 100, marker="o", color="#3b6ea5")
ax.set_xlabel("Hour (JST)")
ax.set_ylabel("Large vehicles (%)")
ax.set_title("Share of large vehicles by hour")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Was it busier than usual?
Compare each day with the median of the same counter on the same weekday in the other weeks.
A ratio well above 1 marks an unusual day: holidays, detours, events.

In [ ]:
daily = (loop.assign(date=loop.time_jst.dt.normalize())
             .groupby(["counter_id", "date"])
             .agg(vehicles=("vehicles", "sum"), hours=("vehicles", "count"),
                  weekday=("weekday", "first"), is_holiday=("is_holiday", "first"))
             .reset_index())
daily = daily[daily.hours == 24].drop(columns="hours")  # full days only
usual = daily.groupby(["counter_id", "weekday"]).vehicles.transform("median")
n_same = daily.groupby(["counter_id", "weekday"]).vehicles.transform("size")
daily["ratio"] = np.where((n_same >= 5) & (usual > 0), daily.vehicles / usual, np.nan)

top = (daily.dropna(subset=["ratio"]).sort_values("ratio", ascending=False)
            .merge(counters[counters.sensor == "loop"][["counter_id", "prefecture_en", "municipality_en"]],
                   on="counter_id", how="left")
            .head(10))
top[["date", "is_holiday", "counter_id", "prefecture_en", "municipality_en", "vehicles", "ratio"]]

In [ ]:
national = daily.groupby("date").ratio.median()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(national.index, national.values, color="#3b6ea5")
ax.axhline(1, color="grey", lw=1)
ax.set_ylabel("Median ratio to a usual day")
ax.set_title("Nationwide: traffic compared with the usual level for that weekday")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("Busiest days nationwide:", ", ".join(f"{d:%Y-%m-%d} ({r:.2f})" for d, r in national.nlargest(5).items()))

The peaks fall in holiday periods: the Marine Day weekend (2026-07-18 to 07-20), the week of
Mountain Day and Obon in August, and the five-day September break (Saturday 2026-09-19 to
Wednesday 09-23). A ratio compares a day with the same weekday, so a weekday holiday stands out
most.

## Busy at rush hour, slow at rush hour
`census_2021_counters.csv` ties 365 permanent counters to their section of the 2021 Road Traffic
Census. For each direction the census gives a travel speed for the rush hours (the slower of 7-9 and
17-19 on an autumn weekday, stops at signals included; mostly from ETC2.0 probe cars, on some sections
filled in or carried over from 2015) and for the daytime hours (9-17). Below, each counter's busiest
rush hour on an average 2026 weekday, per lane, is set against how much its road slowed at rush hour
in 2021.

In [ ]:
census = pd.read_csv(f"{DATA}/census_2021_counters.csv", dtype={"census_section": str})
workday = loop[(loop.time_jst.dt.dayofweek < 5) & ~loop.is_holiday.fillna(False).astype(bool)]
profile = workday.groupby(["counter_id", "hour"]).vehicles.mean().unstack()

speed = census.set_index("counter_id")
speed["rush_kmh"] = speed[["peak_speed_up_kmh", "peak_speed_down_kmh"]].mean(axis=1)
speed["day_kmh"] = speed[["offpeak_speed_up_kmh", "offpeak_speed_down_kmh"]].mean(axis=1)
speed["slowdown"] = 1 - speed.rush_kmh / speed.day_kmh
# Same hours as the census rush: the 7, 8, 17 and 18 o'clock hours, both directions.
speed["rush_per_lane"] = profile[[7, 8, 17, 18]].max(axis=1) / speed.lanes
speed = speed.dropna(subset=["slowdown", "rush_per_lane"])
speed["group"] = pd.qcut(speed.rush_per_lane, 5, labels=False)
print(f"{len(speed)} counters, rank correlation "
      f"{speed.rush_per_lane.corr(speed.slowdown, method='spearman'):.2f}")

In [ ]:
g = speed.groupby("group").agg(lo=("rush_per_lane", "min"), hi=("rush_per_lane", "max"),
                               day=("day_kmh", "mean"), rush=("rush_kmh", "mean"))
g["lost"] = (1 - g.rush / g.day) * 100
x = np.arange(len(g))
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x, g.lost.clip(lower=0), 0.6, color="#c0504d")
for i, r in enumerate(g.itertuples()):
    ax.annotate(f"{int(round(r.lost)) + 0}%\n{r.day:.0f} → {r.rush:.0f} km/h", (i, max(r.lost, 0)),
                xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", fontsize=12)
ax.set_xticks(x, [f"{r.lo:.0f}-{r.hi:.0f}" for r in g.itertuples()])
ax.set_ylim(0, g.lost.max() * 1.35)
ax.set_xlabel("Busiest rush hour on an average 2026 weekday: vehicles per lane (five equal groups)")
ax.set_ylabel("Speed lost at rush hour, 2021 (%)")
ax.set_title("The busier a road is at rush hour, the more it slows at rush hour")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
# Does the link survive within roads of the same size?
for lanes, part in speed[speed.lanes.isin([2, 4])].groupby("lanes"):
    r = part.rush_per_lane.corr(part.slowdown, method="spearman")
    print(f"{lanes:.0f} lanes: {len(part)} counters, rank correlation {r:.2f}")

In [ ]:
# Within one road: did the busier direction slow down more?
dirs = workday.groupby(["counter_id", "hour"])[["up_vehicles", "down_vehicles"]].mean()
dirs = dirs.astype("float64").groupby("counter_id").max()  # busiest rush hour, each direction
both = speed.join(dirs, how="inner")
lean = np.log(both.up_vehicles / both.down_vehicles)
gap = ((1 - both.peak_speed_up_kmh / both.offpeak_speed_up_kmh)
       - (1 - both.peak_speed_down_kmh / both.offpeak_speed_down_kmh))
ok = np.isfinite(lean) & gap.notna()
print(f"{ok.sum()} counters, rank correlation {lean[ok].corr(gap[ok], method='spearman'):.2f}")

The quietest fifth of the counters hardly slows at rush hour; the busiest fifth loses about 15% of
its daytime speed on average. The link holds within two-lane and within four-lane roads, so it is not
just big roads against small ones. It is a comparison between roads: within a single road, the busier
direction was, if anything, slightly less often the one that slowed more (the census may also fill
one direction from the other). The speeds are from 2021 and the counts from 2026, so this says
which roads tend to slow down, not that they were congested on any day here.

## Notes
- Counts, not speeds. The census speeds are one value per road section from autumn 2021,
  so they say which roads tend to slow down, not how fast traffic moved on a given day.
- Values are reference values from the source, not official survey results.
- Source: 「交通量 API（国土交通省）機能による交通量(参考値)」を加工して作成
  （データ提供：公益財団法人日本道路交通情報センター）. Place names: 国土地理院.
- Census: 「令和3年度全国道路・街路交通情勢調査（道路交通センサス）一般交通量調査 箇所別基本表」
  （国土交通省）を加工して作成.
- The archive grows every day: https://github.com/yasumorishima/japan-road-traffic